# 51 · A landmark count-position model (LCP) for PT cross-sections, against scFates, DPT and segment labels

Phase 1 of workstream A diagnosed the coordinate's failures.
- **Mouse within-segment order is method-limited.** A landmark ratio built from selection
  references only agrees with itself at 0.66–0.83 within segments, across disjoint genes and reads.
  scFates and DPT ignore that axis in S1, and scFates also in S3.
- **Human S1 is measurement-limited.**
- **"Depth" is mostly cut size.**
- **P4 is dominated by coverage.**

This notebook fits the pre-registered C1 model, LCP. It is a negative-binomial count likelihood
over a 50-point grid, using landmark genes chosen from external segment references, with the
structure's library as an exposure offset. One rate path per species is shared by all structures,
estimated with equal specimen weight at each grid point (the repeated-object prior). An external
segment-level shape sets orientation and gauge. Segment labels, tissue coordinates, depth and
glomerular distance are never used.

LCP is benchmarked with notebook 36's machinery against an unchanged comparator set:
- SCF13 and DPT13, from notebook 36's cache;
- the zero-cost landmark ratio score (LRS);
- segment labels;
- the segment + coverage oracle.

Protocol: `SCRATCH/ws7/ledger_entry.md`, written before any fit, including its abandonment rules.
Cohort: two male control mice and two cortex sections from one male human donor. Specimens are the
replicates; everything is descriptive.

## 1 · Folders and fixed settings

In [ ]:
import argparse
import json
import math
import os
import sys
from concurrent.futures import ProcessPoolExecutor
from hashlib import sha256
from multiprocessing import get_context
from pathlib import Path

start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
upstream13 = results_root / 'minimal_pt_scfates'
upstream36 = results_root / 'pt_reconstruction_v2' / 'nb36'
external = data_root / 'external'
output_root = results_root / 'pt_reconstruction_v3'
output = output_root / 'nb51'
output.mkdir(parents=True, exist_ok=True)
SAMPLES = {'mouse': ['Ctrl1A2', 'Ctrl1A4'], 'human': ['HUK1_COR1', 'HUK1_MED1']}
source_paths = {n: data_root / 'tubule_by_gene' / f'{n}_tubule_by_gene_caleb.h5ad'
                for names in SAMPLES.values() for n in names}
required = [upstream13 / 'cross_species_pt_scfates.h5ad', upstream13 / 'cross_species_nephron_scfates.h5ad',
            upstream13 / 'ortholog_map_used.csv', upstream36 / 'gene_sets_and_external_contrasts.csv',
            upstream36 / 'p4_heldout_prediction.csv',
            results_root / 'pt_reconstruction_v2' / 'nb35' / 'p2_absorption.csv',
            external / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz',
            external / 'segment_microdissection' / 'GSE56743_RPKM.without.PAseq.txt.gz',
            external / 'census_pt_segments' / 'mouse_pt_segment_pseudobulk_counts.csv.gz',
            external / 'census_pt_segments' / 'human_pt_segment_pseudobulk_counts.csv.gz',
            *source_paths.values()]
for path in required:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

NOTEBOOK_LOGIC_VERSION = '51.landmark_count_position.1'  # Bump when a cached calculation changes.
EARLY_MARKERS, LATE_MARKERS = ('Slc5a2', 'Slc5a12', 'Gatm'), ('Slc22a7', 'Slc7a13', 'Cyp7b1')
N_FOLDS, MIN_DETECTION, ANCHOR_LOOSE = 3, .10, .5   # notebook 36
LANDMARK_THRESHOLD, LANDMARK_MIN_LEVEL = 1.0, 1.0
EQUAL_DEPTH_QUANTILE, SPLIT_EPSILON, SPLIT_SEED = .20, .5, 35   # notebook 36
CALIBRATION_SEED = 51
GRID_SIZE, SPLINE_DF, SHRINK_LAMBDA, MAX_ITER, TOL = 50, 6, 10.0, 100, 1e-4
TEMPERATURES = (1, 2, 4, 8, 16, 32, 64, 128)
MOUSE_SHAPE_POINTS, HUMAN_SHAPE_POINTS = (1 / 6, 1 / 2, 5 / 6), (1 / 3, 5 / 6)
INJECTION_SHARE_MOD, INJECTION_LOG_AMPLITUDE, OWN_INJECTION_SEED = 10, math.log(2), 357
START_SEEDS, START_NOISE = (1, 2, 3), .5
BASIS_DF, P4C_DF = 6, 4
WORKERS = int(os.environ.get('PSEUDOSPACE_51_WORKERS', '24'))
print('Results folder:', output)

## 2 · Inputs: notebook 13's structures and the original counts (as notebook 36)

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import sparse
from scipy.stats import spearmanr

import pseudospace.landmark_position as lcp_module
from pseudospace.anchor_coordinate import (bulk_segment_profiles, census_segment_profiles, conserved_anchor_mask,
                                           positive_control_masks, pt_specific_mask, segment_contrast)
from pseudospace.coordinate_arms import heldout_prediction, registration_gap, within_segment_agreement
from pseudospace.coordinate_evaluation import (absorption_ratio, binomial_thin_to_depth, concordance,
                                               group_rank_correlations, log_normalize, poisson_count_split,
                                               scale_counts, sha256_folds)
from pseudospace.landmark_position import (deviance_gain, fit_lcp, grid_points, landmark_ratio_score,
                                           piecewise_shape, predictive_loglik, project_lcp,
                                           residual_spline_gain, scaled_gain)
from pseudospace.pathway_inputs import rebuild_pt_expression
from pseudospace.stage_cache import cached_payload, code_digest, digest, stage_is_fresh, stage_key
from pseudospace.within_segment_validation import fit_segment_covariate_oracle, predict_segment_covariate_oracle

saved_pt = ad.read_h5ad(upstream13 / 'cross_species_pt_scfates.h5ad', backed='r').obs.copy()
saved_neph = ad.read_h5ad(upstream13 / 'cross_species_nephron_scfates.h5ad', backed='r').obs.copy()
key_files = [p for p in sorted((upstream13 / 'stage_cache').glob('pass2_nephron_embedding__*.h5ad.key.json'))
             if "'logic': '13-nephron-pass2-v1'" in json.loads(p.read_text())['params']]
if len(key_files) != 1:
    raise FileNotFoundError(f'Expected one notebook-13 pass-2 cache, found {len(key_files)}; rerun notebook 13.')
neph = ad.read_h5ad(str(key_files[0])[:-len('.key.json')])
if not neph.obs_names.equals(saved_neph.index):
    raise ValueError('Notebook 13 pass-2 cache does not match its saved nephron structures.')
pt_rows = np.flatnonzero(neph.obs.coarse_class.astype(str).eq('PT').to_numpy())
if not neph.obs_names[pt_rows].equals(saved_pt.index):
    raise ValueError('PT rows differ from notebook 13 saved PT structures.')
rebuilt = rebuild_pt_expression(
    saved_neph[['sample', 'comparison_species', 'shared_pseudospace']].astype({'sample': str, 'comparison_species': str}),
    source_paths, pd.read_csv(upstream13 / 'ortholog_map_used.csv'), target_sum=1e4)
COUNTS = sparse.csr_matrix(rebuilt.layers['counts']).astype(np.int64)
genes_all = pd.Index(rebuilt.var_names)
measured_both = rebuilt.var.measured_in_both_inputs.to_numpy(bool)
cache_genes = pd.Index(neph.var_names)
cache_cols = genes_all.get_indexer(cache_genes)
if (cache_cols < 0).any() or abs(COUNTS[:, cache_cols] - sparse.csr_matrix(neph.layers['counts'])).max() != 0:
    raise ValueError('Rebuilt counts differ from notebook 13 cached counts.')
CONSTRUCTION_GENES = cache_genes[(neph.var.highly_variable_for_harmony & neph.var.highly_variable).to_numpy(bool)]
neph_obs = neph.obs.copy()
del neph

species_pt = saved_pt.comparison_species.astype(str).to_numpy()
specimen_pt = saved_pt['sample'].astype(str).to_numpy()
segment_pt = saved_pt.segment_class.astype(str).str.replace('PT-', '', regex=False).to_numpy()
human_pt = species_pt == 'human'
SCF_SAVED = saved_pt.shared_pseudospace.to_numpy(float)
DPT_SAVED = saved_pt.dpt_pseudospace.to_numpy(float)
PT_COUNTS = COUNTS[pt_rows]
LIBRARY_PT = np.asarray(PT_COUNTS[:, measured_both].sum(axis=1)).ravel().astype(float)
if (LIBRARY_PT <= 0).any():
    raise ValueError('A PT structure has no counts in the shared panel.')
EVAL_ORIGINAL = log_normalize(PT_COUNTS, LIBRARY_PT)
detection = pd.DataFrame({sp: np.asarray((PT_COUNTS[species_pt == sp] > 0).mean(axis=0)).ravel()
                          for sp in SAMPLES}, index=genes_all)
print(f'{len(pt_rows):,} PT structures · {len(genes_all):,} genes · counts reproduce notebook 13')

## 3 · External references, notebook 36's evaluation sets and the landmark panels

- **Selection references** (landmarks only): mouse microdissection, female mouse snRNA, human snRNA
  donor half A.
- **Evaluation references** (metrics only): male mouse snRNA, including its adjacent-segment
  contrasts, and human half B.

A landmark has |S3 − early| ≥ 1 with the same sign in its species' selection references, level ≥ 1,
detection ≥ 10% in that species' PT and PT specificity in that species (notebook 36's rule, per
species). Notebook 36's conserved anchors and evaluation masks are rebuilt unchanged and checked.

In [ ]:
mouse_bulk = pd.read_csv(external / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz', sep='\t')
mouse_micro = bulk_segment_profiles(mouse_bulk.drop_duplicates('Gene_symbol').set_index('Gene_symbol'),
                                    '_PT{seg}_').reindex(genes_all)


def census(label, segment_map, early):
    counts = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
    meta = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_meta.csv', index_col=0)
    return census_segment_profiles(counts, meta.loc[counts.index], segment_map, early=early).reindex(genes_all)


mouse_sn = census('mouse', {'epithelial cell of proximal tubule segment 1': 'S1',
                            'epithelial cell of proximal tubule segment 2': 'S2',
                            'epithelial cell of proximal tubule segment 3': 'S3'}, ['S1', 'S2'])
human_sn = census('human', {'kidney proximal convoluted tubule epithelial cell': 'early',
                            'epithelial cell of proximal tubule segment 3': 'S3'}, None)
human_donors = sorted(set(human_sn.columns.get_level_values('donor')))
HALF_A, HALF_B = human_donors[0::2], human_donors[1::2]
if HALF_A != ['21_015', 'HCA_51_ref', 'PRECISE-34'] or HALF_B != ['HCA_29_ref', 'HCA_55_ref', 'PRECISE-36']:
    raise ValueError(f'Human donor halves differ from the pre-registered split: {HALF_A} / {HALF_B}')


def micro_contrast(frame, late, early):
    return frame[late] - frame[early], frame[[late, early]].max(axis=1)


ext = {'select_mouse_micro': micro_contrast(mouse_micro, 'S3', 'early'),
       'select_mouse_sn_female': segment_contrast(mouse_sn, sex='female'),
       'select_human_sn_halfA': segment_contrast(human_sn, donors=HALF_A),
       'mouse_sn_male': segment_contrast(mouse_sn, sex='male'),
       'mouse_sn_male_S2mS1': segment_contrast(mouse_sn, late='S2', early='S1', sex='male'),
       'mouse_sn_male_S3mS1': segment_contrast(mouse_sn, late='S3', early='S1', sex='male'),
       'mouse_sn_male_S3mS2': segment_contrast(mouse_sn, late='S3', early='S2', sex='male'),
       'mouse_micro': micro_contrast(mouse_micro, 'S3', 'early'),
       'mouse_micro_S2mS1': micro_contrast(mouse_micro, 'S2', 'S1'),
       'mouse_micro_S3mS1': micro_contrast(mouse_micro, 'S3', 'S1'),
       'mouse_micro_S3mS2': micro_contrast(mouse_micro, 'S3', 'S2'),
       'human_sn_halfB': segment_contrast(human_sn, donors=HALF_B)}
contrast = pd.DataFrame({k: v[0] for k, v in ext.items()})
level = pd.DataFrame({k: v[1] for k, v in ext.items()})

classes = neph_obs.coarse_class.astype(str).to_numpy()
specimen_neph = neph_obs['sample'].astype(str).to_numpy()
library_neph = np.asarray(COUNTS[:, measured_both].sum(axis=1)).ravel().astype(float)
normalized = sparse.csr_matrix(COUNTS.multiply((1e4 / library_neph)[:, None]))
class_means = {}
for sp, names in SAMPLES.items():
    columns = {}
    for cls in sorted(set(classes)):
        per = [np.asarray(normalized[(classes == cls) & (specimen_neph == n)].mean(axis=0)).ravel()
               for n in names if ((classes == cls) & (specimen_neph == n)).sum() >= 20]
        columns[cls] = np.mean(per, axis=0)
    class_means[sp] = pd.DataFrame(columns, index=genes_all)
del normalized
eligible_anchor = (pd.Series(measured_both, index=genes_all) & detection.mouse.ge(MIN_DETECTION)
                   & detection.human.ge(MIN_DETECTION) & pt_specific_mask(class_means))
selection = ['select_mouse_micro', 'select_mouse_sn_female', 'select_human_sn_halfA']
A_LOOSE = conserved_anchor_mask(contrast[selection], level[selection], eligible_anchor, threshold=ANCHOR_LOOSE)
markers = pd.Series(genes_all.isin(EARLY_MARKERS + LATE_MARKERS), index=genes_all)
saved36 = pd.read_csv(upstream36 / 'gene_sets_and_external_contrasts.csv', index_col=0)
if not (saved36.anchor_loose_A166rule.astype(bool).reindex(genes_all).fillna(False) == A_LOOSE).all():
    raise ValueError("Notebook 36's conserved anchors do not reproduce.")


def evaluation_mask(sp, reference):
    return (pd.Series(measured_both, index=genes_all) & ~A_LOOSE & ~markers & detection[sp].ge(MIN_DETECTION)
            & level[reference].ge(1) & contrast[reference].notna())


control_eligible = (pd.Series(measured_both, index=genes_all) & ~A_LOOSE & ~markers & detection.mouse.ge(MIN_DETECTION)
                    & detection.human.ge(MIN_DETECTION) & level['mouse_sn_male'].ge(1))
D_OPPOSITE, D_FLAT = positive_control_masks(contrast['human_sn_halfB'], contrast['mouse_sn_male'], contrast['mouse_micro'],
                                            level['human_sn_halfB'], control_eligible)
FOLD = pd.Series(sha256_folds(genes_all, N_FOLDS), index=genes_all)
EVAL_GENES = genes_all[(evaluation_mask('mouse', 'mouse_sn_male') | evaluation_mask('human', 'human_sn_halfB')
                        | evaluation_mask('mouse', 'mouse_micro') | D_OPPOSITE | D_FLAT).to_numpy()]
EVAL_X = EVAL_ORIGINAL[:, genes_all.get_indexer(EVAL_GENES)]
fold_of_eval = FOLD.loc[EVAL_GENES].to_numpy()

# Landmark panels, one per species, from selection references only.
LANDMARK_RULES = {'mouse': ['select_mouse_micro', 'select_mouse_sn_female'], 'human': ['select_human_sn_halfA']}
panel_rows, LANDMARKS = [], {}
for sp, cols in LANDMARK_RULES.items():
    eligible = detection[sp].ge(MIN_DETECTION) & pt_specific_mask({sp: class_means[sp]})
    mask = conserved_anchor_mask(contrast[cols], level[cols], eligible, threshold=LANDMARK_THRESHOLD,
                                 min_level=LANDMARK_MIN_LEVEL)
    genes = list(genes_all[mask.to_numpy()])
    LANDMARKS[sp] = genes
    for g in genes:
        panel_rows.append({'species': sp, 'gene': g, 'sign': int(np.sign(contrast.loc[g, cols[0]])),
                           'fold': int(FOLD[g]), 'contrast_selection': float(contrast.loc[g, cols[0]])})
PANEL = pd.DataFrame(panel_rows)
PANEL.to_csv(output / 'landmark_panel.csv', index=False)
grid = grid_points(GRID_SIZE)
mouse_values = mouse_micro.loc[LANDMARKS['mouse'], ['S1', 'S2', 'S3']].to_numpy(float) * math.log(2)
half_a = human_sn.loc[:, human_sn.columns.get_level_values('donor').isin(HALF_A)]
human_values = np.column_stack([half_a.xs('early', axis=1, level='segment').mean(axis=1).loc[LANDMARKS['human']],
                                half_a.xs('S3', axis=1, level='segment').mean(axis=1).loc[LANDMARKS['human']]]) * math.log(2)
SHAPE = {'mouse': piecewise_shape(MOUSE_SHAPE_POINTS, mouse_values, grid),
         'human': piecewise_shape(HUMAN_SHAPE_POINTS, human_values, grid)}
display(PANEL.groupby(['species', 'sign']).size().unstack().rename(columns={-1: 'early-high', 1: 'late-high'}))
print('Evaluation genes:', len(EVAL_GENES), '· landmarks also in the evaluation set:',
      {sp: int(pd.Index(g).isin(EVAL_GENES).sum()) for sp, g in LANDMARKS.items()})

## 4 · Comparators: SCF13 and DPT13 from notebook 36's cache, unchanged

The full fit must reproduce notebook 13's saved coordinates exactly before any refit is used.

In [ ]:
def load36(stage, spec=None):
    hits = []
    for path in upstream36.joinpath('stage_cache').glob(f'{stage}__*.npz'):
        params = json.loads(Path(str(path) + '.key.json').read_text())['params']
        if spec is None or f"'spec': '{spec}'" in params:
            hits.append(path)
    if len(hits) != 1:
        raise FileNotFoundError(f'Expected one notebook-36 cache for {stage} {spec}, found {len(hits)}.')
    with np.load(hits[0], allow_pickle=False) as stored:
        return {k: stored[k] for k in stored.files}


SPECS = {'full': 'unperturbed', 'fold0': 'fold 0', 'fold1': 'fold 1', 'fold2': 'fold 2', 'ed': 'equal depth 352',
         'half_A': 'split A', 'half_B': 'split B'}
STAGES = {'full': 'arm_scf13_full', 'fold0': 'arm_scf13_fold0', 'fold1': 'arm_scf13_fold1', 'fold2': 'arm_scf13_fold2',
          'ed': 'arm_scf13_ed', 'half_A': 'arm_scf13_half_A', 'half_B': 'arm_scf13_half_B'}
fits36 = {key: load36(STAGES[key], SPECS[key]) for key in SPECS}
if np.abs(fits36['full']['scfates'] - SCF_SAVED).max() > 1e-9 or np.nanmax(np.abs(fits36['full']['dpt'] - DPT_SAVED)) > 1e-6:
    raise ValueError("Notebook 36's cached SCF13/DPT13 do not reproduce notebook 13.")
COORD = {'SCF13': {key: fit['scfates'] for key, fit in fits36.items()},
         'DPT13': {key: fit['dpt'] for key, fit in fits36.items()}}
COORD['SCF13']['loso'] = load36('loso_SCF13')['loso']
print('SCF13 and DPT13 refits loaded and reproduced.')

## 5 · Perturbed counts shared by every arm (notebook 36's seeds) and the LRS comparator

LRS is log((late + 0.5)/(early + 0.5)) over the species' landmark counts. A fold version drops that
fold's landmarks. For metrics that need a [0, 1] scale (splines, gaps), it is mapped to the
within-species pooled rank. That is monotone, so every Spearman metric is unchanged.

In [ ]:
def equal_depth(counts, seed):
    library = np.asarray(counts[:, measured_both].sum(axis=1)).ravel().astype(float)
    target = float(np.quantile(library[pt_rows], EQUAL_DEPTH_QUANTILE))
    return binomial_thin_to_depth(counts, library, target, np.random.default_rng(seed))


ED_352 = equal_depth(COUNTS, 352)
SPLIT_A, SPLIT_B = poisson_count_split(COUNTS, SPLIT_EPSILON, np.random.default_rng(SPLIT_SEED))


def hash_int(gene):
    return int(sha256(str(gene).encode()).hexdigest(), 16)


NONANCHOR_INJECTED = sorted([g for g in CONSTRUCTION_GENES if not A_LOOSE.get(g, False)
                             and g not in EARLY_MARKERS + LATE_MARKERS and hash_int(g) % INJECTION_SHARE_MOD == 0], key=hash_int)


def inject(counts, genes, s0, seed):
    """Human-only exp(±log 2 (2 s0 − 1)) on the listed genes; signs alternate in hash order (notebook 36)."""
    genes = sorted(genes, key=hash_int)
    signs = np.where(np.arange(len(genes)) % 2 == 0, 1.0, -1.0)
    f = INJECTION_LOG_AMPLITUDE * (2 * s0[human_pt] - 1)
    return scale_counts(counts, pt_rows[human_pt], genes_all.get_indexer(genes), f[:, None] * signs[None, :],
                        np.random.default_rng(seed)), genes


INJECTED_36, _ = inject(COUNTS, NONANCHOR_INJECTED, SCF_SAVED, 351)
VARIANTS = {'full': COUNTS, 'ed': ED_352, 'half_A': SPLIT_A, 'half_B': SPLIT_B}


def species_block(counts, sp, genes):
    rows = pt_rows[species_pt == sp]
    block = sparse.csr_matrix(counts[rows][:, genes_all.get_indexer(genes)])
    library = np.asarray(counts[rows][:, measured_both].sum(axis=1)).ravel().astype(float)
    return block, library


def fold_landmarks(sp, k=None):
    return [g for g in LANDMARKS[sp] if k is None or FOLD[g] != k]


def lrs(counts, k=None):
    out = np.full(len(pt_rows), np.nan)
    for sp in SAMPLES:
        genes = fold_landmarks(sp, k)
        sign = PANEL.set_index(['species', 'gene']).loc[sp].sign.loc[genes].to_numpy()
        block, _ = species_block(counts, sp, genes)
        late = np.asarray(block[:, sign > 0].sum(axis=1)).ravel()
        early = np.asarray(block[:, sign < 0].sum(axis=1)).ravel()
        score = landmark_ratio_score(late, early)
        out[species_pt == sp] = (pd.Series(score).rank().to_numpy() - .5) / len(score)
    return out


COORD['LRS'] = {'full': lrs(COUNTS), **{f'fold{k}': lrs(COUNTS, k) for k in range(N_FOLDS)},
                'ed': lrs(ED_352), 'half_A': lrs(SPLIT_A), 'half_B': lrs(SPLIT_B)}
print('LRS built for the full data, three folds, equal depth and both count-split halves.')

## 6 · LCP: temperature calibration, then all fits

The temperature T is chosen per species, once, by count-split predictive likelihood (seed 51,
landmark genes only):
- fit on half A at each T in {1, …, 128};
- score half B's counts under half A's posterior mixture.

T is then frozen. Every fit is cached, and fits run in parallel worker processes.

In [ ]:
cache_root = output / 'stage_cache'
code = digest([NOTEBOOK_LOGIC_VERSION, Path(lcp_module.__file__)])
FIT_PARAMS = dict(grid_size=GRID_SIZE, df=SPLINE_DF, lam=SHRINK_LAMBDA, max_iter=MAX_ITER, tol=TOL)


def _fit_job(job):
    from threadpoolctl import threadpool_limits
    with threadpool_limits(1):
        return fit_lcp(job['counts'], job['library'], job['specimen'], job['shape'], temperature=job['temperature'],
                       init_noise=job.get('init_noise', 0.0), seed=job.get('seed', 0), **FIT_PARAMS)


def run_jobs(jobs):
    """jobs: name -> dict(counts, library, specimen, shape, temperature, ...). Cached per name and input."""
    keys, results, missing = {}, {}, []
    for name, job in jobs.items():
        params = {**FIT_PARAMS, 'temperature': job['temperature'], 'init_noise': job.get('init_noise', 0.0),
                  'seed': job.get('seed', 0)}
        inputs = {k: job[k] for k in ('counts', 'library', 'specimen', 'shape')}
        key = stage_key(f'lcp_{name}', params=params, inputs=inputs, code=code_digest(code))
        keys[name] = (params, inputs)
        if not stage_is_fresh(cache_root / f'lcp_{name}__{key}.npz', key):
            missing.append(name)
    computed = {}
    if missing:
        with ProcessPoolExecutor(max_workers=min(WORKERS, len(missing)), mp_context=get_context('fork')) as pool:
            for name, result in zip(missing, pool.map(_fit_job, [jobs[n] for n in missing])):
                computed[name] = result
    for name in jobs:
        params, inputs = keys[name]
        payload = cached_payload(f'lcp_{name}', lambda name=name: {k: v for k, v in computed[name].items()},
                                 root=cache_root, params=params, inputs=inputs, code=code)
        results[name] = payload
    return results


def job(counts, sp, genes, temperature, rows=None, **extra):
    block, library = species_block(counts, sp, genes)
    spec = specimen_pt[species_pt == sp]
    if rows is not None:
        block, library, spec = block[rows], library[rows], spec[rows]
    idx = [LANDMARKS[sp].index(g) for g in genes]
    return {'counts': block.toarray().astype(np.int64), 'library': library, 'specimen': spec,
            'shape': SHAPE[sp][idx], 'temperature': float(temperature), **extra}


CAL_A, CAL_B = poisson_count_split(COUNTS, .5, np.random.default_rng(CALIBRATION_SEED))
calibration = run_jobs({f'{sp}_calib_T{t}': job(CAL_A, sp, LANDMARKS[sp], t) for sp in SAMPLES for t in TEMPERATURES})
cal_rows = []
for sp in SAMPLES:
    block_b, library_b = species_block(CAL_B, sp, LANDMARKS[sp])
    for t in TEMPERATURES:
        fit = calibration[f'{sp}_calib_T{t}']
        model = {'eta': fit['eta'], 'theta': fit['theta'], 'posterior': fit['posterior']}
        cal_rows.append({'species': sp, 'temperature': t, 'half-B predictive log-likelihood':
                         predictive_loglik(model, block_b.toarray(), library_b),
                         'iterations': int(fit['iterations']), 'converged': bool(fit['converged']),
                         'median posterior entropy': float(np.median(fit['entropy']))})
CALIBRATION = pd.DataFrame(cal_rows)
CALIBRATION.to_csv(output / 'temperature_calibration.csv', index=False)
TEMPERATURE = {sp: int(CALIBRATION[CALIBRATION.species.eq(sp)].set_index('temperature')
                       ['half-B predictive log-likelihood'].idxmax()) for sp in SAMPLES}
display(CALIBRATION.round(2))
print('Selected temperatures (frozen for every refit):', TEMPERATURE)

In [ ]:
main_jobs = {}
for sp in SAMPLES:
    T = TEMPERATURE[sp]
    for variant, counts in VARIANTS.items():
        main_jobs[f'{sp}_{variant}'] = job(counts, sp, LANDMARKS[sp], T)
    for k in range(N_FOLDS):
        main_jobs[f'{sp}_fold{k}'] = job(COUNTS, sp, fold_landmarks(sp, k), T)
    spec = specimen_pt[species_pt == sp]
    for name in SAMPLES[sp]:
        main_jobs[f'{sp}_loso_train_without_{name}'] = job(COUNTS, sp, LANDMARKS[sp], T, rows=spec != name)
    for seed in START_SEEDS:
        main_jobs[f'{sp}_start{seed}'] = job(COUNTS, sp, LANDMARKS[sp], T, init_noise=START_NOISE, seed=seed)
main_jobs['human_injection36'] = job(INJECTED_36, 'human', LANDMARKS['human'], TEMPERATURE['human'])
FITS = run_jobs(main_jobs)


def assemble(key):
    out = np.full(len(pt_rows), np.nan)
    for sp in SAMPLES:
        out[species_pt == sp] = FITS[f'{sp}_{key}']['mean']
    return out


COORD['LCP'] = {key: assemble(key) for key in ('full', 'fold0', 'fold1', 'fold2', 'ed', 'half_A', 'half_B')}
loso = np.full(len(pt_rows), np.nan)
for sp in SAMPLES:
    spec = specimen_pt[species_pt == sp]
    for name in SAMPLES[sp]:
        fit = FITS[f'{sp}_loso_train_without_{name}']
        model = {'eta': fit['eta'], 'theta': fit['theta'], 'temperature': float(fit['temperature']), 'grid': grid}
        block, library = species_block(COUNTS, sp, LANDMARKS[sp])
        rows = spec == name
        target = np.flatnonzero(species_pt == sp)[rows]
        loso[target] = project_lcp(model, block[rows].toarray(), library[rows])['mean']
COORD['LCP']['loso'] = loso

# Own-landmark injection: human-only, 10% of human landmarks, s0 = LCP's own full human position.
OWN_INJECTED_GENES = sorted([g for g in LANDMARKS['human'] if hash_int(g) % INJECTION_SHARE_MOD == 0], key=hash_int)
INJECTED_OWN, _ = inject(COUNTS, OWN_INJECTED_GENES, np.nan_to_num(COORD['LCP']['full']), OWN_INJECTION_SEED)
FITS.update(run_jobs({'human_injection_own': job(INJECTED_OWN, 'human', LANDMARKS['human'], TEMPERATURE['human'])}))
fit_table = pd.DataFrame([{'fit': name, 'iterations': int(f['iterations']), 'converged': bool(f['converged']),
                           'objective decreases': int(f['objective_decreases']), 'temperature': float(f['temperature'])}
                          for name, f in FITS.items()])
fit_table.to_csv(output / 'lcp_fit_diagnostics.csv', index=False)
display(fit_table)

## 7 · Metrics for every arm, per species × segment

- P1 and P1w are fold-protected and scored on **evaluation references only**. For mouse P1w these
  are the male-snRNA adjacent contrasts.
- Notebook 36's microdissection-based P1w is reported for continuity and flagged as circular for LRS
  and LCP, whose landmarks come from microdissection.
- Gene-fold agreement, P5, the library correlation, LOSO, count split and registration gap use
  notebook 36's definitions.

In [ ]:
ARMS = ['SCF13', 'DPT13', 'LRS', 'LCP']
segment_group = np.where(human_pt, np.where(segment_pt == 'S3', 'S3', 'early'), segment_pt)
groups_within = np.char.add(np.char.add(specimen_pt.astype(str), '|'), segment_group.astype(str))
P1W_EVAL = {('mouse', 'S1'): 'mouse_sn_male_S2mS1', ('mouse', 'S2'): 'mouse_sn_male_S3mS1',
            ('mouse', 'S3'): 'mouse_sn_male_S3mS2', ('human', 'early'): 'human_sn_halfB', ('human', 'S3'): 'human_sn_halfB'}
P1W_MICRO = {('mouse', 'S1'): 'mouse_micro_S2mS1', ('mouse', 'S2'): 'mouse_micro_S3mS1', ('mouse', 'S3'): 'mouse_micro_S3mS2'}


def species_r(expression, z, genes):
    keep = np.isfinite(z)
    table = group_rank_correlations(expression[keep], z[keep], specimen_pt[keep])
    table.index = genes
    return pd.DataFrame({sp: table[names].mean(axis=1, skipna=False) for sp, names in SAMPLES.items()})


def within_r(expression, z, genes):
    keep = np.isfinite(z)
    table = group_rank_correlations(expression[keep], z[keep], groups_within[keep])
    table.index = genes
    return table


def protected(arm, function):
    parts = []
    for k in range(N_FOLDS):
        cols = fold_of_eval == k
        parts.append(function(EVAL_X[:, cols], COORD[arm][f'fold{k}'], EVAL_GENES[cols]))
    return pd.concat(parts).loc[EVAL_GENES]


rows, R, RW = [], {}, {}
for arm in ARMS:
    R[arm], RW[arm] = protected(arm, species_r), protected(arm, within_r)
    for sp, ref in (('mouse', 'mouse_sn_male'), ('human', 'human_sn_halfB'), ('mouse', 'mouse_micro')):
        mask = evaluation_mask(sp, ref).loc[EVAL_GENES].to_numpy()
        rows.append({'metric': 'P1', 'arm': arm, 'species': sp, 'segment': 'all', 'reference': ref,
                     'circular for landmark arms': ref == 'mouse_micro',
                     'value': concordance(R[arm].loc[mask, sp], contrast.loc[EVAL_GENES[mask], ref])[0]})
    for refs, tag in ((P1W_EVAL, False), (P1W_MICRO, True)):
        for (sp, seg), ref in refs.items():
            mask = evaluation_mask(sp, ref).loc[EVAL_GENES].to_numpy()
            for n in SAMPLES[sp]:
                rows.append({'metric': 'P1w', 'arm': arm, 'species': sp, 'segment': seg, 'specimen': n, 'reference': ref,
                             'circular for landmark arms': tag,
                             'value': concordance(RW[arm].loc[mask, f'{n}|{seg}'], contrast.loc[EVAL_GENES[mask], ref])[0]})
# Segment labels as a coordinate (P1 only; they carry no within-segment order).
labels_z = pd.Series(segment_pt).map({'S1': 0., 'S2': 1., 'S3': 2.}).to_numpy()
r_labels = species_r(EVAL_X, labels_z, EVAL_GENES)
for sp, ref in (('mouse', 'mouse_sn_male'), ('human', 'human_sn_halfB'), ('mouse', 'mouse_micro')):
    mask = evaluation_mask(sp, ref).loc[EVAL_GENES].to_numpy()
    rows.append({'metric': 'P1', 'arm': 'segment labels', 'species': sp, 'segment': 'all', 'reference': ref,
                 'circular for landmark arms': ref == 'mouse_micro',
                 'value': concordance(r_labels.loc[mask, sp], contrast.loc[EVAL_GENES[mask], ref])[0]})


def cell_rows(metric, arm, table, **extra):
    for rec in table.itertuples():
        sp = 'human' if rec.specimen in SAMPLES['human'] else 'mouse'
        rows.append({'metric': metric, 'arm': arm, 'species': sp, 'segment': rec.segment, 'specimen': rec.specimen,
                     'value': rec.rho, **extra})


log_library = np.log(LIBRARY_PT)
for arm in ARMS:
    c = COORD[arm]
    for a, b in ((0, 1), (0, 2), (1, 2)):
        cell_rows('gene-fold agreement', arm, within_segment_agreement(c[f'fold{a}'], c[f'fold{b}'], specimen_pt, segment_pt),
                  pair=f'{a}-{b}')
    cell_rows('P5 equal depth', arm, within_segment_agreement(c['ed'], c['full'], specimen_pt, segment_pt))
    cell_rows('count split A vs B', arm, within_segment_agreement(c['half_A'], c['half_B'], specimen_pt, segment_pt))
    cell_rows('position vs log library', arm, within_segment_agreement(c['full'], log_library, specimen_pt, segment_pt))
    if 'loso' in c:
        cell_rows('LOSO', arm, within_segment_agreement(c['loso'], c['full'], specimen_pt, segment_pt))
    for key in ('full', 'fold0', 'fold1', 'fold2', 'half_A', 'half_B'):
        gap, _ = registration_gap(c[key], species_pt, specimen_pt, segment_pt)
        for seg, value in gap.items():
            rows.append({'metric': 'registration gap (human − mouse median)', 'arm': arm, 'species': 'both',
                         'segment': seg, 'refit': key, 'value': float(value)})
for seed in START_SEEDS:
    cell_rows('perturbed start vs primary', 'LCP', within_segment_agreement(assemble(f'start{seed}'), COORD['LCP']['full'],
                                                                           specimen_pt, segment_pt), seed=seed)

### P2 · absorption of an injected human-only gradient

- **LCP:** its own landmarks, plus notebook 36's non-anchor genes.
- **LRS:** its own landmarks.
- **SCF13 and DPT13:** reported from notebook 35, unchanged.

Material absorption follows notebook 35's flag: median ratio < 0.85, or human within-segment ρ
(refit against frozen) < 0.90.

In [ ]:
def interaction(expression, z, genes):
    r = species_r(expression, z, pd.Index(genes))
    return (r.human - r.mouse).to_numpy()


p2_rows = []
for label, counts, genes, refit in (('LCP own landmarks', INJECTED_OWN, OWN_INJECTED_GENES, 'human_injection_own'),
                                    ('LCP notebook-36 genes', INJECTED_36, NONANCHOR_INJECTED, 'human_injection36'),
                                    ('LRS own landmarks', INJECTED_OWN, OWN_INJECTED_GENES, None)):
    injected_eval = log_normalize(counts[pt_rows], np.asarray(counts[pt_rows][:, measured_both].sum(axis=1)).ravel().astype(float))
    cols = genes_all.get_indexer(genes)
    arm = label.split()[0]
    z0 = COORD[arm]['full']
    if refit is None:
        z1 = lrs(counts)
    else:
        z1 = z0.copy()
        z1[human_pt] = FITS[refit]['mean']
    original = interaction(EVAL_ORIGINAL[:, cols], z0, genes)
    frozen = interaction(injected_eval[:, cols], z0, genes)
    refitted = interaction(injected_eval[:, cols], z1, genes)
    moved = within_segment_agreement(z1, z0, specimen_pt, segment_pt)
    rec = {'arm': label, **absorption_ratio(original, frozen, refitted, floor=.02),
           'human within-segment rho': float(moved[moved.specimen.isin(SAMPLES['human'])].rho.median())}
    rec['material absorption (notebook 35 flag)'] = bool(rec['median_ratio'] < .85 or rec['human within-segment rho'] < .90)
    rec['median ratio <= 1.10 (as written in the plan)'] = bool(rec['median_ratio'] <= 1.10)
    p2_rows.append(rec)
p2_35 = pd.read_csv(results_root / 'pt_reconstruction_v2' / 'nb35' / 'p2_absorption.csv')
P2 = pd.concat([pd.DataFrame(p2_rows), p2_35.rename(columns={'coordinate': 'arm'}).assign(source='notebook 35')], ignore_index=True)
P2.to_csv(output / 'p2_absorption.csv', index=False)
display(P2)

### P4, P4c and P4c-dev · held-out genes, cross-specimen within species

- Fold-k evaluation genes are predicted from each arm's fold-k-withheld coordinate.
- P4 compares a coordinate spline with segment means (notebook 36).
- P4c adds a within-segment spline of the coordinate to the segment + coverage quadratic oracle.
- P4c-dev uses Poisson deviance with a library offset.

The SCF13 and DPT13 P4 values must reproduce notebook 36.

In [ ]:
covariates = np.column_stack([log_library, np.log(neph_obs.n_spots.to_numpy(float)[pt_rows]),
                              np.log(np.asarray((COUNTS[pt_rows] > 0).sum(axis=1)).ravel())])
anatomy_code = pd.Series(segment_pt).map({'S1': 0, 'S2': 1, 'S3': 2}).to_numpy()
p4_rows = []
for sp, names in SAMPLES.items():
    genes_sp = evaluation_mask(sp, 'mouse_sn_male' if sp == 'mouse' else 'human_sn_halfB').loc[EVAL_GENES].to_numpy()
    for k in range(N_FOLDS):
        cols = genes_sp & (fold_of_eval == k)
        count_cols = genes_all.get_indexer(EVAL_GENES[cols])
        for train, test in ((names[0], names[1]), (names[1], names[0])):
            tr, te = specimen_pt == train, specimen_pt == test
            ytr, yte = EVAL_X[tr][:, cols].toarray(), EVAL_X[te][:, cols].toarray()
            oracle = fit_segment_covariate_oracle(ytr, covariates[tr], anatomy_code[tr], specimen_pt[tr], degree=2)
            base_tr = predict_segment_covariate_oracle(oracle, covariates[tr], anatomy_code[tr])
            base_te = predict_segment_covariate_oracle(oracle, covariates[te], anatomy_code[te])
            ctr, cte = PT_COUNTS[tr][:, count_cols].toarray(), PT_COUNTS[te][:, count_cols].toarray()
            seg_means = {s: ytr[segment_pt[tr] == s].mean(axis=0) for s in ('S1', 'S2', 'S3')}
            seg_pred = np.vstack([seg_means[s] for s in segment_pt[te]])
            var = np.maximum(ytr.var(axis=0), 1e-12)
            quad_sq, seg_sq = (yte - base_te) ** 2, (yte - seg_pred) ** 2
            for seg in ('all', 'S1', 'S2', 'S3'):
                r = np.ones(te.sum(), bool) if seg == 'all' else segment_pt[te] == seg
                p4_rows.append({'arm': 'segment + coverage oracle', 'species': sp, 'fold': k, 'train': train,
                                'segment': seg, 'P4': scaled_gain(seg_sq, quad_sq, var, r)})
            for arm in ARMS:
                z = COORD[arm][f'fold{k}']
                p4 = heldout_prediction(ytr, z[tr], segment_pt[tr], yte, z[te], segment_pt[te], basis_df=BASIS_DF)
                spline = residual_spline_gain(ytr, z[tr], segment_pt[tr], base_tr, yte, z[te], segment_pt[te], base_te,
                                              basis_df=P4C_DF)
                dev = deviance_gain(ctr, LIBRARY_PT[tr], z[tr], segment_pt[tr], cte, LIBRARY_PT[te], z[te], segment_pt[te],
                                    basis_df=P4C_DF)
                for seg in ('all', 'S1', 'S2', 'S3'):
                    r = np.ones(te.sum(), bool) if seg == 'all' else segment_pt[te] == seg
                    p4_rows.append({
                        'arm': arm, 'species': sp, 'fold': k, 'train': train, 'segment': seg,
                        'P4': (1 - np.sum(p4['coordinate_mse'] / np.maximum(p4['train_var'], 1e-12))
                               / np.sum(p4['segment_mse'] / np.maximum(p4['train_var'], 1e-12))) if seg == 'all' else np.nan,
                        'P4c': scaled_gain(spline['base_mse'], spline['coordinate_mse'], spline['train_var'], r),
                        'P4c-dev': float(1 - dev['coordinate_deviance'][r].sum() / dev['segment_deviance'][r].sum())})
P4 = pd.DataFrame(p4_rows)
P4.to_csv(output / 'p4_p4c_heldout_prediction.csv', index=False)
saved_p4 = pd.read_csv(upstream36 / 'p4_heldout_prediction.csv')
check = P4[P4.segment.eq('all') & P4.arm.isin(['SCF13', 'DPT13'])].merge(
    saved_p4, on=['arm', 'species', 'fold', 'train'])
if len(check) != 24 or np.abs(check['P4'] - check['gain vs segment oracle']).max() > 1e-9:
    raise ValueError("SCF13/DPT13 P4 does not reproduce notebook 36.")
print('P4 for SCF13 and DPT13 reproduces notebook 36 exactly (24 cells).')

### Uncertainty (reported, not gated)

This is the fraction of structures whose half-B MAP falls in the half-A 80% posterior interval,
and the reverse, together with the interval widths of the full fit.

In [ ]:
unc_rows = []
for sp in SAMPLES:
    a, b, f = FITS[f'{sp}_half_A'], FITS[f'{sp}_half_B'], FITS[f'{sp}_full']
    seg = segment_pt[species_pt == sp]
    spec = specimen_pt[species_pt == sp]
    for s in ('S1', 'S2', 'S3'):
        for n in SAMPLES[sp]:
            m = (seg == s) & (spec == n)
            unc_rows.append({'species': sp, 'specimen': n, 'segment': s,
                             'half-B MAP in half-A 80% interval': float(np.mean((b['map'][m] >= a['q10'][m]) & (b['map'][m] <= a['q90'][m]))),
                             'half-A MAP in half-B 80% interval': float(np.mean((a['map'][m] >= b['q10'][m]) & (a['map'][m] <= b['q90'][m]))),
                             'median 80% interval width (full)': float(np.median(f['q90'][m] - f['q10'][m])),
                             'median entropy (full)': float(np.median(f['entropy'][m])),
                             'median position (full)': float(np.median(f['mean'][m]))})
UNCERTAINTY = pd.DataFrame(unc_rows)
UNCERTAINTY.to_csv(output / 'lcp_uncertainty.csv', index=False)
display(UNCERTAINTY.round(3))

## 8 · Summary per species × segment, and the pre-registered decision rules

In [ ]:
METRICS = pd.DataFrame(rows)
METRICS.to_csv(output / 'metrics_long.csv', index=False)
cells = METRICS[METRICS.metric.isin(['gene-fold agreement', 'P5 equal depth', 'count split A vs B',
                                     'position vs log library', 'LOSO', 'perturbed start vs primary'])]
by_segment = cells.groupby(['metric', 'species', 'segment', 'arm']).value.median().unstack('arm')
p1w = METRICS[METRICS.metric.eq('P1w')]
p1w_table = p1w.groupby(['circular for landmark arms', 'species', 'segment', 'arm']).value.mean().unstack('arm')
p4_table = P4.groupby(['species', 'segment', 'arm'])[['P4', 'P4c', 'P4c-dev']].mean().unstack('arm')
gap_table = METRICS[METRICS.metric.str.startswith('registration')].assign(absval=lambda d: d.value.abs())
gap_max = gap_table[gap_table.refit.ne('full')].groupby('arm').absval.max()
by_segment.to_csv(output / 'summary_cells_by_segment.csv')
p1w_table.to_csv(output / 'summary_p1w.csv')
p4_table.to_csv(output / 'summary_p4.csv')
display(by_segment.round(3)); display(p1w_table.round(3)); display(p4_table.round(4))


def fold_summary(arm, sp):
    """Mean over segments of the per-segment median gene-fold agreement within one species."""
    t = METRICS[METRICS.metric.eq('gene-fold agreement') & METRICS.arm.eq(arm) & METRICS.species.eq(sp)]
    return float(t.groupby('segment').value.median().mean())


def p1w_summary(arm, sp):
    t = p1w[p1w.arm.eq(arm) & p1w.species.eq(sp) & ~p1w['circular for landmark arms']]
    return float(t.value.mean())


def p4c_cells(arm, sp):
    return P4[P4.arm.eq(arm) & P4.species.eq(sp) & P4.segment.eq('all')].P4c.to_numpy()


def p1(arm, sp):
    ref = 'mouse_sn_male' if sp == 'mouse' else 'human_sn_halfB'
    return float(METRICS[METRICS.metric.eq('P1') & METRICS.arm.eq(arm) & METRICS.species.eq(sp)
                         & METRICS.reference.eq(ref)].value.iloc[0])


p5 = METRICS[METRICS.metric.eq('P5 equal depth')]
summary = {arm: {**{f'gene-fold {sp}': fold_summary(arm, sp) for sp in SAMPLES},
                 **{f'P1w {sp}': p1w_summary(arm, sp) for sp in SAMPLES},
                 **{f'P1 {sp}': p1(arm, sp) for sp in SAMPLES},
                 **{f'P4c mean {sp}': float(p4c_cells(arm, sp).mean()) for sp in SAMPLES},
                 **{f'P4c > 0 cells {sp}': int((p4c_cells(arm, sp) > 0).sum()) for sp in SAMPLES},
                 **{f'P4 mean {sp}': float(P4[P4.arm.eq(arm) & P4.species.eq(sp) & P4.segment.eq('all')].P4.mean()) for sp in SAMPLES},
                 'P5 overall': float(p5[p5.arm.eq(arm)].value.median()),
                 'P5 human S1': float(p5[p5.arm.eq(arm) & p5.species.eq('human') & p5.segment.eq('S1')].value.mean()),
                 'registration gap': float(gap_max[arm]),
                 'LOSO': float(METRICS[METRICS.metric.eq('LOSO') & METRICS.arm.eq(arm)].value.median())
                 if (METRICS.metric.eq('LOSO') & METRICS.arm.eq(arm)).any() else np.nan}
           for arm in ARMS}
SUMMARY = pd.DataFrame(summary)
SUMMARY.to_csv(output / 'summary_by_arm.csv')
display(SUMMARY.round(4))

L, D, S_, Z = SUMMARY['LCP'], SUMMARY['DPT13'], SUMMARY['SCF13'], SUMMARY['LRS']
abandon = {
    '(i) mouse gene-fold <= DPT13': bool(L['gene-fold mouse'] <= D['gene-fold mouse']),
    '(ii) mouse P1w <= DPT13 (evaluation references)': bool(L['P1w mouse'] <= D['P1w mouse']),
    '(iii) mouse P4c > 0 in fewer than 5 of 6 cells': bool(L['P4c > 0 cells mouse'] < 5),
    '(iv) mouse P4c mean <= LRS or mouse gene-fold <= LRS': bool(L['P4c mean mouse'] <= Z['P4c mean mouse']
                                                                  or L['gene-fold mouse'] <= Z['gene-fold mouse'])}
own = P2[P2.arm.eq('LCP own landmarks')].iloc[0]
adopt = {
    '(A) P1 >= SCF13 - 0.03, both species': bool(all(L[f'P1 {sp}'] >= S_[f'P1 {sp}'] - .03 for sp in SAMPLES)),
    '(A) P5 >= 0.80 overall and >= 0.70 human S1': bool(L['P5 overall'] >= .80 and L['P5 human S1'] >= .70),
    '(A) registration gap <= 0.15': bool(L['registration gap'] <= .15),
    '(A) no material P2 absorption (own landmarks)': not bool(own['material absorption (notebook 35 flag)']),
    '(B) P4c > 0 in >= 5/6 mouse and >= 4/6 human cells': bool(L['P4c > 0 cells mouse'] >= 5 and L['P4c > 0 cells human'] >= 4),
    '(B) P4c mean above best of SCF13, DPT13, LRS (both species)': bool(all(
        L[f'P4c mean {sp}'] > max(S_[f'P4c mean {sp}'], D[f'P4c mean {sp}'], Z[f'P4c mean {sp}']) for sp in SAMPLES)),
    '(B) P1w >= DPT13 + 0.05 mouse, >= DPT13 human': bool(L['P1w mouse'] >= D['P1w mouse'] + .05 and L['P1w human'] >= D['P1w human']),
    '(B) gene-fold >= 0.70 mouse, >= 0.59 human': bool(L['gene-fold mouse'] >= .70 and L['gene-fold human'] >= .59),
    '(B) LOSO >= 0.85': bool(L['LOSO'] >= .85)}
DECISION = {'abandonment rules (any true -> abandon)': abandon, 'abandon': any(abandon.values()),
            'adoption criteria before the downstream gate': adopt,
            'adopt pending downstream gate': (not any(abandon.values())) and all(adopt.values()),
            'temperatures': TEMPERATURE, 'landmarks': {sp: len(g) for sp, g in LANDMARKS.items()}}
print(json.dumps(DECISION, indent=1))

## 9 · The LCP coordinate for workstream B

The file has the same columns as `pt_reconstruction_v2/primary_pt_coordinate.csv`, plus the
posterior summaries. The position is the posterior mean on the 50-point grid. It is a gauge set by
external segment shapes, not physical length.

In [ ]:
lcp_frame = pd.DataFrame({'structure_id': saved_pt.index, 'specimen': specimen_pt, 'species': species_pt,
                          'segment_class': saved_pt.segment_class.astype(str).to_numpy(),
                          'position': COORD['LCP']['full'], 'coordinate_method': 'LCP',
                          'method_description': 'notebook 51 landmark count-position model: NB likelihood with library '
                                                'offset over selection-reference landmarks, specimen-balanced shared rate '
                                                'paths per species, tempered posterior over a 50-point grid'})
for column, key in (('posterior_q10', 'q10'), ('posterior_q90', 'q90'), ('posterior_entropy', 'entropy'),
                    ('posterior_map', 'map')):
    values = np.full(len(pt_rows), np.nan)
    for sp in SAMPLES:
        values[species_pt == sp] = FITS[f'{sp}_full'][key]
    lcp_frame[column] = values
lcp_frame['temperature'] = [TEMPERATURE[sp] for sp in species_pt]
if lcp_frame[['position', 'posterior_q10', 'posterior_q90']].isna().any().any():
    raise ValueError('LCP coordinate has missing values.')
lcp_frame.to_csv(output_root / 'lcp_coordinate.csv', index=False)
with open(output / 'summary.json', 'w') as handle:
    json.dump({'logic_version': NOTEBOOK_LOGIC_VERSION, 'decision': DECISION,
               'summary_by_arm': SUMMARY.round(6).to_dict(), 'p2': P2.drop(columns=['source'], errors='ignore').to_dict('records'),
               'n_fits': len(FITS)}, handle, indent=1, default=str)
print('Saved', output_root / 'lcp_coordinate.csv')